# 5.1. Multilayer Perceptrons
D2L의 Multilayer Perceptrons장을 PyTorch 기준으로 정리함.

- 선형 모델이 복잡한 관계를 표현하기 어려운 이유
- 은닉층의 역할
- 층을 여러 개 쌓기만 해서는 표현력이 증가하지 않는 이유
- 비선형 활성화 함수가 필요한 이유
- ReLU, Sigmoid, Tanh 함수의 특징과 미분
- MLP가 복잡한 함수를 근사할 수 있는 이유

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

## 1. 선형 모델에서 신경망으로

앞에서는 softmax regression을 사용해서 Fashion-MNIST이미지를 분류했다.

Softmax regression은 입력을 출력에 직접 연결하는 선형 모델이다.

$$
\mathbf{O} = \mathbf{XW} + \mathbf{b}
$$

이런 선형 모델은 입력과 출력 사이의 관계가 단순한 경우에 효과적이다.

이미지, 음성, 자연어처럼 복잡한 데이터에는 입력 특성 사이의 상호작용이 존재한다. 그래서 입력과 출력 사이의 관계를 하나의 선형 변환만으로 표현하기 어렵다.

## 2. 선형 모델의 한계

선형 모델에서는 하나의 입력 특성이 증가할 때 출력이 항상 같은 방향으로 변한다.

가중치가 양수라면 입력이 증가할수록 출력도 증가하고, 가중치가 음수라면 입력이 증가할수록 출력은 감소한다.

선형모델은 기본적으로 다음과 같은 관계를 가정한다.

$$
y = wx + b
$$

하지만 현실의 데이터는 이러한 관계를 따르지 않는 경우가 많다.

예를 들어서 체온과 건강 위험도의 관계를 생각해보자.

체온이 정상 범위보다 높아지거나 낮아지면 위험도가 증가한다.

따라서 체온과 위험도의 관계는 단순히 계속 증가하거나 계속 감소하는 선형 관계가 아니다.

이미지 데이터에서는 문제가 더 복잡하다.

특정 픽셀 하나의 밝기만으로 고양이와 강아지를 구분할 수 없다. 특정 픽셀의 의미는 주변 픽셀, 윤곽선, 모양 등과 함께 결정된다.

따라서 신경망은 단순히 입력을 출력으로 변환하는 것뿐만 아니라, 입력으로부터 유용한 표현을 함께 학습해야 한다.

## 3. 은닉층

선형 모델의 한계를 극복하기 위해서 입력층과 출력층 사이에 하나 이상의 은닉층을 추가할 수 있다.

$$
\text{입력층}
\rightarrow
\text{은닉층}
\rightarrow
\text{출력층}
$$

여러 개의 완전연결층을 쌓아 만든 신경망을 다층 퍼셉트론이라고 한다.

다층 퍼셉트론은 영어로 Multilayer Perceptron이다. 일반적으로 MLP라고 부른다.

입력층은 데이터를 전달할 뿐 실제 파라미터 계산을 수행하지 않기 때문에 일반적으로 신경망의 층 수에 포함하지 않는다.

아까 본 이 구조는 계산을 수행하는 층이 은닉층, 출력층 두 개이기 때문에 2층신경망이다.

$$
\text{입력층}
\rightarrow
\text{은닉층}
\rightarrow
\text{출력층}
$$

## 4. 완전 연결층

완전연결층에서는 이전 층의 모든 뉴런이 다음 층의 모든 뉴런과 연결된다.

입력 특성이 $d$개이고 은닉 뉴런이 $h$개라면, 은닉층의 가중치는 다음 shape를 가진다.

$$
\mathbf{W}^{(1)} \in \mathbb{R}^{d \times h}
$$

출력 클래스가 $q$개라면 출력층 가중치는 다음 shape을 가진다.

$$
\mathbf{W}^{(2)} \in \mathbb{R}^{h \times q}
$$

미니배치를 
